# Deep-Live-Cam Arx - Colab Test

This notebook validates the Linux/Python application, dependencies, imports, models, and tests. An optional final section tests a face swap together with a 2.5D image background on one uploaded image.

**Windows result is not guaranteed:** Colab cannot execute `.bat`/PowerShell, `winget`, virtual-environment activation, desktop shortcuts, or Windows DLL behavior.

In [ ]:
import os

REPO_URL = "https://github.com/Arxdevmm/deep-live-cam-arx.git"
REPO_DIR = "/content/deep-live-cam-arx"
COMMIT = ""  # Optional, for example: 15ae0a4

if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
else:
    %cd /content
    !git clone --depth 1 {REPO_URL}
    %cd {REPO_DIR}

if COMMIT:
    !git fetch --depth 1 origin {COMMIT}
    !git checkout {COMMIT}

print("Repository:", os.getcwd())

In [ ]:
!python --version
!git --no-pager status --short --branch

In [ ]:
!python -m pip install --upgrade pip setuptools wheel
!python -m pip install -r requirements.txt
!python -m pip install pytest

In [ ]:
!python -m compileall -q run.py modules tests
!pytest -q

In [ ]:
from modules import model_downloader

for name in ("gfpgan-1024.onnx", "inswapper_128_fp16.onnx"):
    path = model_downloader.ensure_model(name)
    print(f"{name}: {path or 'FAILED'}")

In [ ]:
!python run.py --version

## Optional: Face swap + 2.5D background on one image

This runs an offline image test in the same order as the live preview: replace the target's background, then apply the face swap. It does not test the webcam UI or GLB/glTF scenes. The first run downloads the rembg segmentation model, the optional depth model, and face-swap models. Use fictional faces or real faces only with informed consent.

In [ ]:
%pip install "rembg>=2.0.87"

In [ ]:
from pathlib import Path

import cv2
from google.colab import files

INPUT_DIR = Path("/content/dlc_background_test")
INPUT_DIR.mkdir(parents=True, exist_ok=True)

def upload_image(role, prompt):
    print(prompt)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one image for each prompt.")
    filename, content = next(iter(uploaded.items()))
    path = INPUT_DIR / f"{role}_{Path(filename).name}"
    path.write_bytes(content)
    if cv2.imread(str(path)) is None:
        raise ValueError(f"Could not decode uploaded image: {path.name}")
    return str(path)

source_face_path = upload_image("source_face", "Upload one source-face image.")
target_image_path = upload_image("target", "Upload one target portrait image with a visible face.")
background_image_path = upload_image("background", "Upload one replacement background image.")

In [ ]:
import cv2
from IPython.display import display
from PIL import Image

from modules.background_3d import Background3DEngine

target = cv2.imread(target_image_path)
background_engine = Background3DEngine()
if not background_engine.available:
    raise RuntimeError(background_engine.error or "Background segmentation is unavailable.")

background_only_path = str(INPUT_DIR / "target_with_background.png")
background_result = background_engine.process(
    target,
    background_image_path,
    parallax_strength=0.35,
    perspective_strength=0.25,
    blur_strength=0.05,
)
depth_available = background_engine.depth_available
depth_error = background_engine.depth_error
background_engine.close()
if background_result is None or not cv2.imwrite(background_only_path, background_result):
    raise RuntimeError("Could not save the background-replaced target image.")
if not depth_available:
    print("Depth model unavailable; the perspective/parallax fallback was used.")
    if depth_error:
        print(depth_error)

display(Image.open(background_only_path))

In [ ]:
import subprocess
import sys

import onnxruntime as ort

providers = set(ort.get_available_providers())
execution_provider = "cuda" if "CUDAExecutionProvider" in providers else "cpu"
final_output_path = INPUT_DIR / "faceswap_with_background.png"
subprocess.run(
    [
        sys.executable,
        "run.py",
        "--execution-provider",
        execution_provider,
        "--source",
        source_face_path,
        "--target",
        background_only_path,
        "--output",
        str(final_output_path),
    ],
    check=True,
)
if not final_output_path.is_file() or final_output_path.stat().st_size == 0:
    raise RuntimeError("The face-swap command did not produce an output image.")
print(f"Combined result: {final_output_path}")

In [ ]:
import matplotlib.pyplot as plt
from google.colab import files

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for axis, label, path in zip(
    axes,
    ("Target", "Background replacement", "Face swap + background"),
    (target_image_path, background_only_path, str(final_output_path)),
):
    axis.imshow(Image.open(path))
    axis.set_title(label)
    axis.axis("off")
plt.tight_layout()
plt.show()
files.download(str(final_output_path))

## Pass criteria

The notebook is successful only if:

1. dependency installation exits successfully;
2. `compileall` reports no syntax errors;
3. `pytest -q` has no failures;
4. both model checks print paths rather than `FAILED`;
5. `python run.py --version` prints the application version.

After these pass, test `install.bat` on a real Windows machine with no old copy and with winget available.